# 1B. Construção da base do Cenário B

Este notebook prepara uma versão ampliada da base analítica,
incorporando informações desagregadas das taxas de evasão segundo
sexo, PPI, faixa etária e deficiência.

A variável-alvo permanece sendo a taxa de evasão no período t.
As variáveis adicionais são utilizadas com defasagem temporal (t-1).

In [1]:
from pathlib import Path
import sys
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "src").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.preprocessing import preparar_indicador

In [5]:
CAMINHO_BASE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "INDIC_UF_2010_2024.xlsx"
)

df_evasao = preparar_indicador(
    CAMINHO_BASE,
    "TX_EVASAO",
    "evasao",
    possui_deficiencia=True
)

print(df_evasao.shape)

(378, 16)


In [6]:
CAMINHO_BASE_CENARIO_A = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "base_modelo_uf.csv"
)

df_cenario_a = pd.read_csv(
    CAMINHO_BASE_CENARIO_A
)

print(df_cenario_a.shape)

(189, 10)


In [7]:
colunas_desagregadas = [
    "ano_fluxo",
    "uf",

    "evasao_feminino",
    "evasao_masculino",

    "evasao_ppi_sim",
    "evasao_ppi_nao",

    "evasao_ate_19",
    "evasao_20_22",
    "evasao_23_24",
    "evasao_25_29",
    "evasao_30_39",
    "evasao_40_49",
    "evasao_50_mais",

    "evasao_deficiencia_sim",
    "evasao_deficiencia_nao"
]

In [8]:
df_desagregado = df_evasao[
    colunas_desagregadas
].copy()

In [9]:
df_desagregado["ano_inicio"] = (
    df_desagregado["ano_fluxo"]
    .str[:4]
    .astype(int)
)

In [10]:
df_desagregado = df_desagregado.sort_values(
    ["uf", "ano_inicio"]
).reset_index(drop=True)

In [11]:
colunas_para_defasar = [
    coluna
    for coluna in colunas_desagregadas
    if coluna not in ["ano_fluxo", "uf"]
]

for coluna in colunas_para_defasar:
    df_desagregado[f"{coluna}_t_1"] = (
        df_desagregado
        .groupby("uf")[coluna]
        .shift(1)
    )

In [13]:
colunas_t1 = [
    "uf",
    "ano_fluxo"
] + [
    f"{coluna}_t_1"
    for coluna in colunas_para_defasar
]

df_desagregado_t1 = df_desagregado[
    colunas_t1
].copy()

In [14]:
df_cenario_b = df_cenario_a.merge(
    df_desagregado_t1,
    on=["uf", "ano_fluxo"],
    how="inner"
)

df_cenario_b.shape

(189, 23)

In [15]:
df_cenario_b.isna().sum()

ano_fluxo                     0
uf                            0
evasao_t                      0
conclusao_t                   0
retencao_t                    0
permanencia_t                 0
evasao_t_1                    0
conclusao_t_1                 0
retencao_t_1                  0
permanencia_t_1               0
evasao_feminino_t_1           0
evasao_masculino_t_1          0
evasao_ppi_sim_t_1            0
evasao_ppi_nao_t_1            0
evasao_ate_19_t_1             0
evasao_20_22_t_1              0
evasao_23_24_t_1              0
evasao_25_29_t_1              0
evasao_30_39_t_1              0
evasao_40_49_t_1              0
evasao_50_mais_t_1            0
evasao_deficiencia_sim_t_1    0
evasao_deficiencia_nao_t_1    0
dtype: int64

In [16]:
df_cenario_b.duplicated(
    ["uf", "ano_fluxo"]
).sum()

np.int64(0)

In [17]:
df_cenario_b["uf"].nunique()

27

In [18]:
df_cenario_b.columns.tolist()

['ano_fluxo',
 'uf',
 'evasao_t',
 'conclusao_t',
 'retencao_t',
 'permanencia_t',
 'evasao_t_1',
 'conclusao_t_1',
 'retencao_t_1',
 'permanencia_t_1',
 'evasao_feminino_t_1',
 'evasao_masculino_t_1',
 'evasao_ppi_sim_t_1',
 'evasao_ppi_nao_t_1',
 'evasao_ate_19_t_1',
 'evasao_20_22_t_1',
 'evasao_23_24_t_1',
 'evasao_25_29_t_1',
 'evasao_30_39_t_1',
 'evasao_40_49_t_1',
 'evasao_50_mais_t_1',
 'evasao_deficiencia_sim_t_1',
 'evasao_deficiencia_nao_t_1']

In [19]:
CAMINHO_CENARIO_B = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "base_modelo_uf_cenario_b.csv"
)

df_cenario_b.to_csv(
    CAMINHO_CENARIO_B,
    index=False
)

print(
    f"Base do Cenário B salva em:\n{CAMINHO_CENARIO_B}"
)

Base do Cenário B salva em:
/home/sara/Documentos/tcc-evasao-ensino-superior/data/processed/base_modelo_uf_cenario_b.csv


In [20]:
pd.read_csv(
    CAMINHO_CENARIO_B
).shape

(189, 23)